# Lecture 25: Binary classification and predictive maintenance
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/natrask/engr1050-fall2026/blob/main/NewMaterial/Lecture24_Nov18/lec24.ipynb)

## Overview

In the last lecture, we learned how to use PyTorch and neural networks for **regression** - predicting continuous values like concrete strength. Today, we'll learn about **classification** - predicting discrete categories like whether a material will fracture under a given load.

We'll start with the simplest classification algorithm: **logistic regression**. We'll start by making a model that performs **binary classification**: given an input, it will predict a True/False relationship. We will use this to step through a **predictive maintenance** problem where we can use machine learning to diagnose whether a machine is broken.

Today's section will be a self-guided tour through how classification works and how it maps onto more complicated problems. Practice using Gemini to get explanations on code, and ask Prof. Trask or the TA as soon as you have questions. We will assume that you have read and understood this notebook as we step through the final homework on Monday.

In [ ]:
# Import necessary libraries
import torch
import matplotlib.pyplot as plt
import numpy as np

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("PyTorch version:", torch.__version__)

# Review: Our template for building a PyTorch model

We now have a cookbook for how to build and train a model. There are two steps.
- Use `modelClass` to define a model
- Update the training loop to use gradient descent to train your model

We have done this for several models now
- Linear regression
- You extended that to polynomial regression
- Fitting stress/strain models to experimental data
- Training a multilayer perceptron (MLP)

Today will be our last model, where we learn to use an MLP to do binary classification. To make this convenient, I provide below a recipe that you can use whenever you want to fit a model to data. I provide TODO comments to make it clear what steps you need to fill in to make this work.

To get one last round of practice filling out this cookbook, today we will plug an MLP into the modelclass with a sigmoid on the end. Then we will define a *binary cross-entropy loss* which is a new kind of loss function that is used to train binary random variables.

### TODOs to define your model

``` python
class myModelClass:
    def __init__(self, a0, b0): # TODO - replace a0,b0 with initial guesses for parameters

        # TODO - Define trainable parameters
        # Example:
            # self.a = torch.tensor(a0, requires_grad=True)
            # self.b = torch.tensor(b0, requires_grad=True)
            # self.c = torch.tensor(0.0, requires_grad=True)
    
    def forward(self, x):
        # Given input x, write formula to evaluate the output of the model

        # TODO - Define forward pass
    # Example:
        # return self.a * x + self.b +  self.c * x**2
```

### TODOs for training loop

```python
# Initialize your model
model = myModelClass(input_dim=1, hidden_dim=40, output_dim=1)

# Training parameters
learning_rate = 0.001
num_steps = 20000
batch_size = 10

for step in range(num_steps):
    # Zero gradients at the start of each step
    # TODO 1: Zero out gradients of each variable in the model
    # Example: 
    # if model.W1.grad is not None:
    #     model.W1.grad.zero_()
    #     model.b1.grad.zero_()
    #     model.W2.grad.zero_()
    #     model.b2.grad.zero_()
    
    # Accumulate loss as a tensor
    total_loss = torch.tensor(0.0, requires_grad=True)
    
    # Randomly select batch_size data points
    for j in range(batch_size):
        # Randomly access each x,y pair in the dataset
        i = torch.randint(0, len(x_data), (1,)).item()
        x = torch.tensor([x_data[i].item()], dtype=torch.float32)
        y_true = torch.tensor([y_data[i].item()], dtype=torch.float32)
        
        # TODO 2: evaluate the loss function 
        # Example:
            # y_pred = model.forward(x)
            # loss = (y_pred - y_true)**2

        total_loss = total_loss + loss.sum()
    
    # Backward pass on the accumulated loss
    total_loss.backward()
    
    # Update parameters using gradient descent
    # TODO 3: replace model parameters with each one
    # Example:
        # model.W1 -= learning_rate * model.W1.grad
        # model.b1 -= learning_rate * model.b1.grad
        # model.W2 -= learning_rate * model.W2.grad
        # model.b2 -= learning_rate * model.b2.grad
```

## Classification vs Regression

OK - let's get to it. I'm summarizing here the difference between classification and regression problems. Make sure you're comfortable with the distinction.

| **Regression** | **Classification** |
|----------------|-------------------|
| Predict continuous values | Predict discrete categories |
| Example: Concrete strength (20.5 MPa) | Example: Material passes quality control (yes/no) |
| Output: Any real number | Output: Category label (0, 1, 2, ...) |

## The Sigmoid Function

For **binary classification** (two classes: 0 or 1), we need to output a probability that our input is `True` - for example "Is a fair coinflip heads?" should return a probability of 0.5. 

To make a neural network output a probability, we need a function function that maps any real number to a probability between 0 and 1. 
- We will take an MLP to map from any input to a score (called a **logit**)
- We feed that logit through a function that maps and number to $[0,1]$

The **sigmoid function** does exactly this:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Properties of the sigmoid:
- $\sigma(z) \in (0, 1)$ for all $z$
- $\sigma(0) = 0.5$
- As $z \to \infty$, $\sigma(z) \to 1$
- As $z \to -\infty$, $\sigma(z) \to 0$

In [ ]:
# Visualize the sigmoid function
z = torch.linspace(-10, 10, 200)
sigma_z = torch.sigmoid(z)

plt.figure(figsize=(8, 5))
plt.plot(z.numpy(), sigma_z.numpy(), 'b-', linewidth=2, label=r'$\sigma(z) = \frac{1}{1 + e^{-z}}$')
plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Decision boundary (P=0.5)')
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.3)
plt.xlabel('z (input)', fontsize=12)
plt.ylabel(r'$\sigma(z)$ (probability)', fontsize=12)
plt.title('Sigmoid Function', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()
plt.ylim(-0.1, 1.1)
plt.show()

## Engineering Application: Material Failure Prediction

Let's create a synthetic dataset simulating a materials engineering problem:

**Scenario**: We're testing aluminum samples under tension. Each sample is subjected to a certain stress level, and we record whether it **failed** (1) or **survived** (0).

- **Input feature $x$**: Applied stress (MPa)
- **Output label $y$**: Failure (1) or survival (0)

In [ ]:
# Generate synthetic material failure data
np.random.seed(42)
torch.manual_seed(42)

# Number of samples
n_samples = 200

# Generate stress values (200-400 MPa)
stress = np.random.uniform(200, 400, n_samples)

# True failure probability increases with stress
# Using a sigmoid-like relationship centered around 300 MPa
true_prob = 1 / (1 + np.exp(-(stress - 300) / 5))

# Generate binary outcomes based on true probability
failure = (np.random.rand(n_samples) < true_prob).astype(float)

# Convert to PyTorch tensors
X = torch.tensor(stress, dtype=torch.float32).reshape(-1, 1)
y = torch.tensor(failure, dtype=torch.float32).reshape(-1, 1)

# Normalize stress for stable training (z-score scaling)
X_mean = X.mean()
X_std = X.std()
X_normalized = (X - X_mean) / X_std

def normalize_stress(x_tensor):
    """Apply the same normalization used during training."""
    return (x_tensor - X_mean) / X_std

print(f"Dataset size: {len(X)} samples")
print(f"Stress range: [{X.min():.1f}, {X.max():.1f}] MPa")
print(f"Failure rate: {y.mean():.2%}")
print(f"Normalized stress: mean={X_normalized.mean():.2f}, std={X_normalized.std(unbiased=False):.2f}")

In [ ]:
# Visualize the dataset
plt.figure(figsize=(10, 6))
plt.scatter(X[y.squeeze()==0].numpy(), y[y.squeeze()==0].numpy(), 
            alpha=0.6, s=50, c='green', label='Survived (y=0)', marker='o')
plt.scatter(X[y.squeeze()==1].numpy(), y[y.squeeze()==1].numpy(), 
            alpha=0.6, s=50, c='red', label='Failed (y=1)', marker='x')
plt.xlabel('Applied Stress (MPa)', fontsize=12)
plt.ylabel('Outcome (0=Survived, 1=Failed)', fontsize=12)
plt.title('Material Failure Dataset', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.yticks([0, 1])
plt.show()

## Binary classification

Unlike regression, where we stuck a linear layer at the end of the hidden layers to make it spit out a constant, now we can stick a sigmoid at the end. 
- The job of the hidden layers is to map the input to a *logit* - something like a score that approaches infinity if yes and negative infinity if no
- The job of the sigmoid is to convert the logit score into a *probability* - the percent chance that the input is either true or false

For example, if we had just one hidden layer, it might look like the following

$$P(y=1 \mid x) = \sigma(wx + b) = \frac{1}{1 + e^{-(wx + b)}}$$

where:
- $x$ is the input (stress)
- $w$ is the **weight** (learnable parameter)
- $b$ is the **bias** (learnable parameter)
- $\sigma$ is the sigmoid function

## Binary Classification Model

In the code below, I took the multilayer perceptron from the last class and modified it with a sigmoid so that is outputs a probability.

This is a **one line modification** of the MLP from last lecture. Take a look at the line marked `NEW` which is the modification to feed the output through a sigmoid.

In [ ]:
class BinaryClassificationModel:
    """Simple logistic regression model for binary classification."""
    def __init__(self, input_dim, hidden_dim, output_dim):
        # Initialize a 2-layer MLP
        # input_dim - dimension of input (e.g., 1 for scalar input)
        # hidden_dim - number of neurons in hidden layer
        # output_dim - dimension of output (e.g., 1 for scalar output)
        
        # Layer 1: input -> hidden
        self.W1 = torch.randn(hidden_dim, input_dim, requires_grad=True)
        self.b1 = torch.randn(hidden_dim, requires_grad=True)
        
        # Layer 2: hidden -> output
        self.W2 = torch.randn(output_dim, hidden_dim, requires_grad=True)
        self.b2 = torch.randn(output_dim, requires_grad=True)
    
    def forward(self, x):
        # Layer 1: compute h1 = tanh(W1*x + b1)
        z1 = self.W1 @ x + self.b1   # Linear transformation
        h1 = torch.tanh(z1)          # Activation function
        
        # Layer 2: compute y = W2*h1 + b2
        y = self.W2 @ h1 + self.b2   # Linear transformation (no activation on output)

        # NEW: Apply sigmoid to output to get probability
        output_probability = torch.sigmoid(y)
        
        return output_probability

model = BinaryClassificationModel(1,5,1)

# Create model instance

## Binary Cross-Entropy Loss

For classification, we can't use MSE loss. Instead, we use **binary cross-entropy** loss:

$$\text{BCE}(y, \hat{y}) = -\left[y \log(\hat{y}) + (1-y) \log(1-\hat{y})\right]$$

where:
- $y \in \{0, 1\}$ is the true label
- $\hat{y} \in (0, 1)$ is the predicted probability

**Intuition**: 
- If $y=1$ (failed), we want $\hat{y} \approx 1$, so $-\log(\hat{y})$ is small
- If $y=0$ (survived), we want $\hat{y} \approx 0$, so $-\log(1-\hat{y})$ is small
- Wrong predictions are heavily penalized (logarithm goes to infinity)

The following code will generate a plot of the cross entropy loss. You can see the two scenarios where the true label is either 1 or 0. The binary cross entropy is a function that is huge when the wrong label is predicted and zero when the right label is predicted. If we train this loss with stochastic gradient descent, we will "push" the model away from wrong classifications.

In [ ]:
# Visualize binary cross-entropy loss
p_hat = torch.linspace(0.001, 0.999, 100)  # Predicted probabilities

# Loss for y=1 (true label is 1)
loss_y1 = -torch.log(p_hat)

# Loss for y=0 (true label is 0)
loss_y0 = -torch.log(1 - p_hat)

plt.figure(figsize=(10, 5))
plt.plot(p_hat.numpy(), loss_y1.numpy(), 'r-', linewidth=2, label='y=1 (true label is "failed")')
plt.plot(p_hat.numpy(), loss_y0.numpy(), 'g-', linewidth=2, label='y=0 (true label is "survived")')
plt.xlabel(r'Predicted Probability $\hat{y}$', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Binary Cross-Entropy Loss', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0, 5)
plt.show()

We can wrap the code to evaluate the cross entropy loss in a function - that will make it easier to write down our training loop. There is something a little tricky in the entropy - the log of zero is infinite, and that will make our code explode if our model predicts either a 0 or a 1 exactly. We can avoid getting any infinities by redefining the input to the log so its always a tiny bit bigger than zero and less than 1.

In [ ]:
def binary_cross_entropy(y_pred, y_true):
    """
    Compute binary cross-entropy loss.
    Args:
        y_pred: Predicted probabilities (batch_size, 1)
        y_true: True labels 0 or 1 (batch_size, 1)
    """
    epsilon = 1e-7  # Small constant to avoid log(0)
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon) # Makes sure predictions are within (0+eps,1-eps) so log doesn't explode
    loss = -(y_true * torch.log(y_pred) + (1 - y_true) * torch.log(1 - y_pred))
    return loss

## Training the Model

We'll use **mini-batch gradient descent** to train the model. This is exactly the same as last week - just a few changes:
- Our data should all take y values of either 0 (false) or 1 (true)
- We need to swap out the MLP model for one that outputs a number between 0 and 1
- We need to swap out the loss for the cross entropy loss

Same as always, we'll do the following
1. Split data into small batches
2. For each batch:
   - Compute predictions
   - Compute loss
   - Backpropagate gradients
   - Update parameters

**Be careful:** Neural networks will only train if data is scaled to be one-ish in magnitude.

To make today's exercise easy, I filled in the TODOs for you.

In [ ]:
# Training parameters
learning_rate = 0.01
num_steps = 2000
batch_size = 10

# Create a new model
model = BinaryClassificationModel(input_dim=1, hidden_dim=30, output_dim=1)

# Track loss history
loss_history = []

for step in range(num_steps):
    step_loss = 0.0

    # Zero out the gradients
    # TODO 1: Zero out gradients of each variable in the model
    if model.W1.grad is not None:
        model.W1.grad.zero_()
        model.b1.grad.zero_()
        model.W2.grad.zero_()
        model.b2.grad.zero_()

    # Accumulate loss as a tensor
    total_loss = torch.tensor(0.0, requires_grad=True)

    # Randomly select batch_size data points
    for j in range(batch_size):
        idx = np.random.randint(len(X))
        xdata = X_normalized[idx] # get data using the **normalized** input
        ydata = y[idx]
        y_pred = model.forward(xdata)
        
        # TODO 2: evaluate the loss function 
        loss = binary_cross_entropy(y_pred, ydata)
        total_loss = total_loss + loss

    # Take gradient over whole minibatch with autograd
    total_loss.backward()

    # Update parameters using gradient descent
    # TODO 3: replace model parameters with each one
    with torch.no_grad():
        model.W1 -= learning_rate * model.W1.grad
        model.b1 -= learning_rate * model.b1.grad
        model.W2 -= learning_rate * model.W2.grad
        model.b2 -= learning_rate * model.b2.grad

    print(f"Step {step}: Loss over minibatch = {total_loss.detach().numpy()/batch_size}")

In [ ]:
# Visualize class boundary after training with data overlaid

plt.figure(figsize=(10, 6))
# Plot data points
plt.scatter(X[y.squeeze() == 0].numpy(), y[y.squeeze() == 0].numpy(),
            alpha=0.6, s=50, c='green', label='Survived (y=0)', marker='o')
plt.scatter(X[y.squeeze() == 1].numpy(), y[y.squeeze() == 1].numpy(),
            alpha=0.6, s=50, c='red', label='Failed (y=1)', marker='x')

# Plot model predictions
x_range = torch.linspace(200, 400, 200).reshape(-1, 1)
y_range = []
for xin in x_range:
    newy = model.forward((xin-X_mean)/X_std).detach().numpy() # Need to normalize the input
    y_range.append(newy)
plt.plot(x_range.numpy(), y_range, 'b-', linewidth=2, label='Model Prediction')
plt.xlabel('Applied Stress (MPa)', fontsize=12)
plt.ylabel('Predicted Probability of Failure', fontsize=12)
plt.title('Model Predictions After Training', fontsize=14)
plt.legend()
plt.ylim(-0.1, 1.1)
plt.grid(True, alpha=0.3)
plt.show()

# Real-World Example: Bearing Vibration Fault Detection and Predictive Diagnostics

Now we're going to show a real-life example where this can be used for a very non-trivial problem. Oftentimes, we can collect sensor readings off of a instrument as it runs and ask whether it is functioning properly or not. While we would prefer to be able to derive an ODE model of a system and use physics to predict exactly what it should be doing, the reality is that these models may be too difficult to derive or solve. Instead, we can attempt to diagnose a failure from signals running off a machine, either in terms of its power consumption, sound (we can all tell from a kathunk-a-thunk whether our dryer is broken or we have a flat tire), or other sensors readings.

We can use machine learning to throw up a red flag that says a machine needs fixing, even if we don't understand exactly how or why it isn't working right. In a world where factory machine downtime can translate to millions of dollars of lost production, these automated systems can quickly diagnose and dispatch solutions - this is a welcome alternative to a human being squinting at page after page of squiggly line readouts from a machine.

In today's exercise we will take data from the Case Western Reserve University Bearing Data Center:

**Source**: Case Western Reserve University Bearing Data Center  
**URL**: http://csegroups.case.edu/bearingdatacenter/home  
**Accessed**: November 16, 2025 

### Problem setup

A ball bearing is a component used to minimize the friction between a rotating shaft, like the drivetrain on a car motor, and the supports that hold it up (e.g. where the driveshaft meets the wheels). The inner cylinder rotates freely within the outer cylinder, rolling on the spherical balls to avoid friction.

<div align="center">
    <img src="../_shared/Images/ballbearing.jpg" alt="Lab Setup" width="300">
</div>


In this experiment, the scientists at Case Western used a motor to drive a ball bearing under load, taking accelerometer measurements off of the bearing to characterize forces as a function of time.
<div align="center">
    <img src="../_shared/Images/CWRU_bearingsetup.jpeg" alt="Lab Setup" width="600">
</div>

They did this for two setups: in one, they had a perfectly functioning ball bearing. In the other, they intentionally introduced a defect by damaging the bearing.

### Download the data

The following code will load these two time series from the internet and save them in the arrays `signal_normal` (for the undamaged reading) and `signal_fault` (for the damaged reading).

In [ ]:
# Download bearing vibration data
import urllib.request
import scipy.io

# Normal bearing
url_normal = 'https://engineering.case.edu/sites/default/files/97.mat'
urllib.request.urlretrieve(url_normal, 'normal.mat')
data_normal = scipy.io.loadmat('normal.mat')

# Faulty bearing (inner race defect)
url_fault = 'https://engineering.case.edu/sites/default/files/105.mat'
urllib.request.urlretrieve(url_fault, 'fault.mat')
data_fault = scipy.io.loadmat('fault.mat')

# Extract vibration signals
signal_normal = data_normal['X097_DE_time'].flatten()
signal_fault = data_fault['X105_DE_time'].flatten()

print(f"Normal bearing: {len(signal_normal)} samples")
print(f"Faulty bearing: {len(signal_fault)} samples")

### Visualize your data

The duration of the experiments is pretty long - there are a ton of readings packed in here, since the sensor can read at 12 kHz (that means 12000 samples per second!). We can visualize both by plotting a few milliseconds to get a sense of what they both look like.

In [ ]:
# Visualize: Normal vs Faulty vibration
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Plot first 0.05 seconds
fs = 12000  # Sampling rate (Hz)
n_samples = 600
time = np.arange(n_samples) / fs

ax1.plot(time, signal_normal[:n_samples], 'g-', linewidth=1)
ax1.set_title('NORMAL Bearing - Smooth Vibration', fontsize=14, fontweight='bold', color='green')
ax1.set_ylabel('Acceleration (g)', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(-1, 1)

ax2.plot(time, signal_fault[:n_samples], 'r-', linewidth=1)
ax2.set_title('FAULTY Bearing - Spiky Vibration', fontsize=14, fontweight='bold', color='red')
ax2.set_xlabel('Time (seconds)', fontsize=12)
ax2.set_ylabel('Acceleration (g)', fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(-1, 1)

plt.tight_layout()
plt.show()

From this we can see that there is a signal in there that may be possible to catch. 
- The peaks of the normal bearing are typical under 0.25 g, while for the faulty they spike up to 1 g
- The wave length of the normal is longer and has rounded smooth peaks and valleys, while the faulty one is spikey and short wavelength (presumably as balls are skipping over the damaged surface).

Our goal is to write a classifier that can map onto a short window of reading and output a probability that the machine is damaged. **Be careful though!** Just because we will see this is a straightforward problem to approach with machine learning, we could just as easily identify if its damaged or not with an if statement that evaluates whether the acceleration is over 0.3 g.

### Set up the binary classification model

We can use our usual code, but now we'll set up the problem so that:
- The input is a window of 20 continuous accelerometer readings
- The output is either a 0 if it came from the healthy machine or a 1 if it was damaged

The only changes we will need are to:
- Modify the classification network to take an input of 20 numbers
- As usual, choose a good `hidden_dim`, `learning_rate` and `num_steps`
- Set up minibatching so that at each step of training, we randomly grab either the broken or unbroken data, and then choose a random set of 20 steps from it

In [ ]:
# Training parameters
learning_rate = 0.01
num_steps = 20000
batch_size = 1  # we hardcode to take a single batch

# Create a new model
model = BinaryClassificationModel(input_dim=20, hidden_dim=100, output_dim=1)

# Track loss history
loss_history = []

for step in range(num_steps):
    step_loss = 0.0
    # Zero out the gradients
    # TODO 1: Zero out gradients of each variable in the model
    if model.W1.grad is not None:
        model.W1.grad.zero_()
        model.b1.grad.zero_()
        model.W2.grad.zero_()
        model.b2.grad.zero_()

    # Accumulate loss as a tensor
    total_loss = torch.tensor(0.0, requires_grad=True)
    
    # TODO 2: evaluate the loss function 
    # For this minibatch, randomly grab either the normal (y = 0) or faulty (y = 1) data set
    # Randomly choose normal (0) or faulty (1) bearing
    is_fault = np.random.randint(0, 2)  # Returns 0 or 1
    
    if is_fault == 0:
        # Use normal bearing data
        signal = signal_normal
        label = 0
    else:
        # Use faulty bearing data
        signal = signal_fault
        label = 1

    # Extract a random 20-sample window from the signal
    start_idx = np.random.randint(0, len(signal) - 20)
    xdata = torch.tensor(signal[start_idx:start_idx+20], dtype=torch.float32)
    ydata = torch.tensor([label], dtype=torch.float32)
    
    # For this minibatch, take a forward pass of the model and add it into step_loss
    y_pred = model.forward(xdata)
    loss = binary_cross_entropy(y_pred, ydata)
    total_loss = total_loss + loss

    # Take gradient over whole minibatch with autograd
    total_loss.backward()

    # Update parameters using gradient descent
    # TODO 3: replace model parameters with each one
    with torch.no_grad():
        model.W1 -= learning_rate * model.W1.grad
        model.b1 -= learning_rate * model.b1.grad
        model.W2 -= learning_rate * model.W2.grad
        model.b2 -= learning_rate * model.b2.grad

    print(f"Step {step}: Loss over minibatch = {total_loss.detach().numpy()/batch_size}")

To check whether our model worked, we can sweep over the data and apply our model over contiguous chunks of 20 sensor readings. We can then plot to see how well our machine-learned sensor predicts failure.

**First.** Write a function that slides the network over all of the input accelerometer values for different times to generate `p_normal` and `p_fault` - predictions of probability failure at a given time. 

In [ ]:
# Prompt: Apply trained model as a sliding window filter to generate probability time series
window_size = 20
stride = 1          # set >1 to downsample
fs = 12000          # sampling rate (already defined above)

def window_probs(signal, window_size, stride):
    probs = []
    centers = []
    for i in range(0, len(signal) - window_size + 1, stride):
        w = torch.tensor(signal[i:i+window_size], dtype=torch.float32)
        # Forward pass (model expects length-20 tensor)
        p = model.forward(w).detach().numpy()
        probs.append(p)
        centers.append(i + window_size/2)
    probs = np.array(probs)
    t = np.array(centers) / fs
    return t, probs

t_normal, p_normal = window_probs(signal_normal, window_size, stride)
t_fault, p_fault   = window_probs(signal_fault, window_size, stride)

**Second.** Visualize these predictions alongside the raw signal to see how well your neural network performs.

In [ ]:
# Prompt: visualize probabilities in 1x2 row above raw signals plotted in 1x2 row below
decimate = 500  # take every 500th window probability for clarity
p_norm_dec = p_normal[::decimate]
t_norm_dec = t_normal[::decimate]
p_fault_dec = p_fault[::decimate]
t_fault_dec = t_fault[::decimate]

# Choose a raw segment length that matches displayed probability time span
# (map window center time range shown in decimated probabilities)
t_span_norm = (t_norm_dec[0], t_norm_dec[-1])
t_span_fault = (t_fault_dec[0], t_fault_dec[-1])

# Convert times back to sample indices
start_norm_idx = int(t_span_norm[0] * fs)
end_norm_idx   = int(t_span_norm[1] * fs)
start_fault_idx = int(t_span_fault[0] * fs)
end_fault_idx   = int(t_span_fault[1] * fs)

# Clip indices to signal bounds
end_norm_idx = min(end_norm_idx, len(signal_normal))
end_fault_idx = min(end_fault_idx, len(signal_fault))

fig, axes = plt.subplots(2, 2, figsize=(14, 6), sharex='col')

# Top-left: Normal probabilities
axes[0,0].plot(t_norm_dec, p_norm_dec, color='green', lw=1.2)
axes[0,0].set_title('Normal: Predicted Fault Probability', color='green')
axes[0,0].set_ylabel('P(fault)')
axes[0,0].grid(alpha=0.3)
axes[0,0].set_ylim(-0.1,1.1)

# Top-right: Faulty probabilities
axes[0,1].plot(t_fault_dec, p_fault_dec, color='red', lw=1.2)
axes[0,1].set_title('Faulty: Predicted Fault Probability', color='red')
axes[0,1].grid(alpha=0.3)
axes[0,1].set_ylim(-0.1,1.1)

# Bottom-left: Raw normal signal segment
time_norm_segment = np.arange(start_norm_idx, end_norm_idx) / fs
axes[1,0].plot(time_norm_segment, signal_normal[start_norm_idx:end_norm_idx], color='green', lw=0.8)
axes[1,0].set_title('Normal: Raw Vibration Segment', color='green')
axes[1,0].set_xlabel('Time (s)')
axes[1,0].set_ylabel('Amplitude')
axes[1,0].grid(alpha=0.3)

# Bottom-right: Raw faulty signal segment
time_fault_segment = np.arange(start_fault_idx, end_fault_idx) / fs
axes[1,1].plot(time_fault_segment, signal_fault[start_fault_idx:end_fault_idx], color='red', lw=0.8)
axes[1,1].set_title('Faulty: Raw Vibration Segment', color='red')
axes[1,1].set_xlabel('Time (s)')
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Submit today's work 

Today we will *not* be collecting exercises. Instead, take time to read through this document carefully and make sure you understand the logic behind all of the pieces. For some of you, this lecture will be crucial for getting your final projects working. For others, this is a good final review to prepare for our last exam. You should be comfortable understanding:

- How classification is different from regression
- Why the model for classification looks different from Monday's model
- How to modify the model and training loop to use a different model/loss
